# Capitulo 01. Modelo de demanda agregada simulado

Este notebook acompana el capitulo 1 de **Macroeconomia Computacional con Python**. Usa datos simulados para mostrar como un relato macroeconomico se convierte en un modelo reproducible.

Advertencia: los datos no son observaciones reales. No deben usarse como evidencia empirica.

In [ ]:
from pathlib import Path
from datetime import date
import csv
import html

In [ ]:
def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "books" / "macroeconomia_computacional_python").exists():
            return candidate
    raise FileNotFoundError("No se encontro la raiz del proyecto.")

ROOT = find_project_root(Path.cwd().resolve())
BOOK = ROOT / "books" / "macroeconomia_computacional_python"
PROCESSED = BOOK / "data" / "processed"
FIGURES = BOOK / "outputs" / "figures"
TABLES = BOOK / "outputs" / "tables"
REPORTS = BOOK / "outputs" / "reports"

for path in [PROCESSED, FIGURES, TABLES, REPORTS]:
    path.mkdir(parents=True, exist_ok=True)

str(BOOK.relative_to(ROOT))

## 1. Parametros del modelo

El modelo resume consumo, inversion, gasto publico y exportaciones netas. Los parametros son pedagogicos y se documentan en el reporte de metadata.

In [ ]:
BASE = {
    "C0": 45.0,
    "c": 0.72,
    "t": 0.18,
    "I": 55.0,
    "G": 70.0,
    "X0": 18.0,
    "m": 0.16,
    "lambda_ajuste": 0.35,
}

def equilibrio(params: dict) -> float:
    numerador = params["C0"] + params["I"] + params["G"] + params["X0"]
    denominador = 1 - params["c"] * (1 - params["t"]) + params["m"]
    return numerador / denominador

def multiplicador_gasto(params: dict) -> float:
    return 1 / (1 - params["c"] * (1 - params["t"]) + params["m"])

round(equilibrio(BASE), 2), round(multiplicador_gasto(BASE), 2)

## 2. Escenarios

Cada escenario cambia pocos parametros para que la interpretacion sea transparente.

In [ ]:
escenarios = {
    "base": {},
    "shock_fiscal": {"G": BASE["G"] + 20.0},
    "shock_fiscal_mayor_apertura": {"G": BASE["G"] + 20.0, "m": 0.28},
    "shock_fiscal_mayor_consumo": {"G": BASE["G"] + 20.0, "c": 0.82},
}

def combinar(base: dict, cambios: dict) -> dict:
    params = base.copy()
    params.update(cambios)
    return params

multiplicadores = []
for nombre, cambios in escenarios.items():
    params = combinar(BASE, cambios)
    multiplicadores.append({
        "escenario": nombre,
        "C0": params["C0"],
        "c": params["c"],
        "t": params["t"],
        "I": params["I"],
        "G": params["G"],
        "X0": params["X0"],
        "m": params["m"],
        "multiplicador_gasto": multiplicador_gasto(params),
        "producto_equilibrio": equilibrio(params),
    })

multiplicadores

## 3. Simulacion dinamica

La produccion se ajusta gradualmente al equilibrio de cada escenario. Esto no representa una estimacion empirica, solo una regla pedagogica de transicion.

In [ ]:
def simular_trayectoria(nombre: str, params: dict, periodos: int = 16) -> list[dict]:
    y_equilibrio = equilibrio(params)
    y = equilibrio(BASE) * 0.96
    registros = []
    for periodo in range(periodos):
        if periodo > 0:
            y = y + params["lambda_ajuste"] * (y_equilibrio - y)
        consumo = params["C0"] + params["c"] * (1 - params["t"]) * y
        impuestos = params["t"] * y
        importaciones = params["m"] * y
        xn = params["X0"] - importaciones
        registros.append({
            "escenario": nombre,
            "periodo": periodo,
            "producto": y,
            "producto_equilibrio": y_equilibrio,
            "consumo": consumo,
            "inversion": params["I"],
            "gasto_publico": params["G"],
            "exportaciones_netas": xn,
            "impuestos": impuestos,
            "importaciones": importaciones,
        })
    return registros

trayectorias = []
for nombre, cambios in escenarios.items():
    trayectorias.extend(simular_trayectoria(nombre, combinar(BASE, cambios)))

trayectorias[:3]

## 4. Exportacion de resultados

In [ ]:
data_path = PROCESSED / "chapter_01_escenarios_demanda_agregada.csv"
table_path = TABLES / "chapter_01_multiplicadores.csv"
figure_path = FIGURES / "chapter_01_trayectorias_producto.svg"
report_path = REPORTS / "chapter_01_metadata_simulacion.md"

def escribir_csv(path: Path, rows: list[dict]) -> None:
    campos = list(rows[0].keys())
    with path.open("w", newline="", encoding="utf-8") as archivo:
        writer = csv.DictWriter(archivo, fieldnames=campos)
        writer.writeheader()
        writer.writerows(rows)

def puntos_svg(rows: list[dict], x0: int, y0: int, width: int, height: int, ymin: float, ymax: float) -> str:
    max_periodo = max(row["periodo"] for row in rows)
    puntos = []
    for row in rows:
        x = x0 + (row["periodo"] / max_periodo) * width
        y = y0 + height - ((row["producto"] - ymin) / (ymax - ymin)) * height
        puntos.append(f"{x:.1f},{y:.1f}")
    return " ".join(puntos)

def escribir_svg(path: Path, rows: list[dict]) -> None:
    colores = {
        "base": "#1f77b4",
        "shock_fiscal": "#2ca02c",
        "shock_fiscal_mayor_apertura": "#d62728",
        "shock_fiscal_mayor_consumo": "#9467bd",
    }
    width, height = 920, 540
    x0, y0, plot_w, plot_h = 80, 55, 720, 390
    ymin = min(row["producto"] for row in rows) * 0.98
    ymax = max(row["producto"] for row in rows) * 1.02
    escenarios_ordenados = list(colores.keys())
    lineas = [
        f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}" viewBox="0 0 {width} {height}">',
        '<rect width="100%" height="100%" fill="#ffffff"/>',
        '<text x="80" y="32" font-family="Arial" font-size="22" font-weight="700">Trayectorias simuladas del producto</text>',
        f'<line x1="{x0}" y1="{y0 + plot_h}" x2="{x0 + plot_w}" y2="{y0 + plot_h}" stroke="#444"/>',
        f'<line x1="{x0}" y1="{y0}" x2="{x0}" y2="{y0 + plot_h}" stroke="#444"/>',
    ]
    for i in range(6):
        y = y0 + i * plot_h / 5
        valor = ymax - i * (ymax - ymin) / 5
        lineas.append(f'<line x1="{x0}" y1="{y:.1f}" x2="{x0 + plot_w}" y2="{y:.1f}" stroke="#e8e8e8"/>')
        lineas.append(f'<text x="20" y="{y + 4:.1f}" font-family="Arial" font-size="12">{valor:.0f}</text>')
    for escenario in escenarios_ordenados:
        datos = [row for row in rows if row["escenario"] == escenario]
        puntos = puntos_svg(datos, x0, y0, plot_w, plot_h, ymin, ymax)
        color = colores[escenario]
        lineas.append(f'<polyline fill="none" stroke="{color}" stroke-width="3" points="{puntos}"/>')
        for punto in puntos.split():
            x, y = punto.split(',')
            lineas.append(f'<circle cx="{x}" cy="{y}" r="3" fill="{color}"/>')
    for idx, escenario in enumerate(escenarios_ordenados):
        y = 95 + idx * 28
        color = colores[escenario]
        etiqueta = html.escape(escenario.replace('_', ' '))
        lineas.append(f'<rect x="825" y="{y - 11}" width="16" height="16" fill="{color}"/>')
        lineas.append(f'<text x="848" y="{y + 2}" font-family="Arial" font-size="13">{etiqueta}</text>')
    lineas.extend([
        '<text x="380" y="505" font-family="Arial" font-size="14">Periodo simulado</text>',
        '<text x="18" y="260" font-family="Arial" font-size="14" transform="rotate(-90 18 260)">Producto</text>',
        '</svg>',
    ])
    path.write_text("\n".join(lineas), encoding="utf-8")

escribir_csv(data_path, trayectorias)
escribir_csv(table_path, multiplicadores)
escribir_svg(figure_path, trayectorias)

parametros_base = "\n".join([f"- `{clave}`: {valor}" for clave, valor in BASE.items()])
metadata = f"""# Metadata de simulacion - Capitulo 1

- Libro: Macroeconomia Computacional con Python
- Capitulo: 01. Del relato macroeconomico al modelo computacional
- Fecha de generacion: {date.today().isoformat()}
- Tipo de datos: simulados, no observados
- Proposito: docencia sobre multiplicador, escenarios y trazabilidad
- Archivo de datos: `{data_path.relative_to(ROOT)}`
- Tabla: `{table_path.relative_to(ROOT)}`
- Figura: `{figure_path.relative_to(ROOT)}`

## Parametros base

{parametros_base}

## Advertencia

Los resultados no son evidencia empirica ni estimaciones causales. Son consecuencias del modelo y parametros declarados.
"""
report_path.write_text(metadata, encoding="utf-8")

[str(path.relative_to(ROOT)) for path in [data_path, table_path, figure_path, report_path]]